### 1. Imports and Configuration

In [7]:
import os
import sys
import random
import platform
from pathlib import Path

import numpy as np
import torch

# -----------------------------------------------------------------------
# Vocabulary -- 15 tokens total.
# Digits 0-9 map directly to their integer value.
# Three special sequence tokens cover FizzBuzz output.
# SEP separates numbers within a sequence; PAD fills unused positions.
# -----------------------------------------------------------------------

VOCAB_SIZE   = 15
TOK_FIZZ     = 10     # FizzBuzz output: divisible by 3 only
TOK_BUZZ     = 11     # FizzBuzz output: divisible by 5 only
TOK_FZBZ     = 12     # FizzBuzz output: divisible by both 3 and 5
TOK_SEP      = 13     # Separator between encoded numbers in a sequence
TOK_PAD      = 14     # Padding token -- ignored by loss and attention

# -----------------------------------------------------------------------
# Sequence -- fixed length for all samples, matching the target
# deployment context (128-token window).
# -----------------------------------------------------------------------

SEQ_LEN      = 128

# -----------------------------------------------------------------------
# Model -- small enough to train in minutes on CPU, large enough to
# show meaningful capacity differences between FP32 and ternary.
# ~25k parameters total.
# -----------------------------------------------------------------------

EMBED_DIM    = 32     # Token embedding and hidden state dimension
NUM_HEADS    = 2      # Attention heads (16 dims per head)
NUM_LAYERS   = 2      # Transformer block count
FF_DIM       = 64     # Feed-forward inner dimension
DROPOUT      = 0.1

# -----------------------------------------------------------------------
# Ternary quantization -- weights with absolute value below the
# threshold are mapped to 0; the rest are mapped to sign(w).
# A small threshold keeps most weights active at initialisation.
# -----------------------------------------------------------------------

TERNARY_THRESH = 0.05

# -----------------------------------------------------------------------
# Training -- both FP32 and ternary models use identical hyper-
# parameters so the comparison is controlled.
# -----------------------------------------------------------------------

BATCH_SIZE   = 64
EPOCHS       = 300
LR           = 3e-4
WEIGHT_DECAY = 1e-4
EVAL_EVERY   = 10     # Compute per-task accuracy every N epochs
SEED         = 42

# -----------------------------------------------------------------------
# Dataset -- 50k samples split equally across four tasks.
# 90/10 train-validation split.
# -----------------------------------------------------------------------

N_SAMPLES    = 10000
TRAIN_FRAC   = 0.9
TASK_NAMES   = ['fibonacci', 'fizzbuzz', 'parity', 'primes']

# -----------------------------------------------------------------------
# Filesystem -- outputs are written relative to the repository root.
# -----------------------------------------------------------------------

ROOT_DIR     = Path('..').resolve()
DATA_DIR     = ROOT_DIR / 'data'
CKPT_DIR     = ROOT_DIR / 'checkpoints'
METRICS_DIR  = ROOT_DIR / 'metrics'

for d in [DATA_DIR, CKPT_DIR, METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------
# Reproducibility -- seed everything before any random operation.
# -----------------------------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f'Vocab size      : {VOCAB_SIZE}  (digits 0-9, FIZZ, BUZZ, FZBZ, SEP, PAD)')
print(f'Sequence length : {SEQ_LEN}')
print(f'Embed dim       : {EMBED_DIM}  |  Heads: {NUM_HEADS}  |  Layers: {NUM_LAYERS}  |  FF: {FF_DIM}')
print(f'Ternary thresh  : {TERNARY_THRESH}')
print(f'Batch size      : {BATCH_SIZE}  |  Epochs: {EPOCHS}  |  LR: {LR}  |  WD: {WEIGHT_DECAY}')
print(f'Eval every      : {EVAL_EVERY} epochs')
print(f'Dataset size    : {N_SAMPLES}  |  Train fraction: {TRAIN_FRAC}')
print(f'Tasks           : {TASK_NAMES}')
# print(f'Data dir        : {DATA_DIR}')
# print(f'Checkpoint dir  : {CKPT_DIR}')
# print(f'Metrics dir     : {METRICS_DIR}')

Vocab size      : 15  (digits 0-9, FIZZ, BUZZ, FZBZ, SEP, PAD)
Sequence length : 128
Embed dim       : 32  |  Heads: 2  |  Layers: 2  |  FF: 64
Ternary thresh  : 0.05
Batch size      : 64  |  Epochs: 300  |  LR: 0.0003  |  WD: 0.0001
Eval every      : 10 epochs
Dataset size    : 10000  |  Train fraction: 0.9
Tasks           : ['fibonacci', 'fizzbuzz', 'parity', 'primes']


### 2. Environment Verification

In [8]:
import importlib

# -----------------------------------------------------------------------
# Verify that all required packages are importable before any notebook
# does real work. Fail loudly here rather than mid-experiment.
# -----------------------------------------------------------------------

REQUIRED_PACKAGES = ['torch', 'numpy', 'matplotlib', 'tqdm', 'sympy']

print('Package availability:')
all_ok = True

for pkg in REQUIRED_PACKAGES:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, '__version__', 'unknown')
        print(f'  {pkg:<15} {ver}')
    except ImportError:
        print(f'  {pkg:<15} NOT FOUND -- run: pip install {pkg}')
        all_ok = False

print()

# -----------------------------------------------------------------------
# Hardware context -- this experiment runs on CPU intentionally.
# MPS and CUDA are detected and reported but not used: running ternary
# ops on GPU tensor cores silently falls back to float arithmetic,
# defeating the purpose of the quantization comparison.
# -----------------------------------------------------------------------

DEVICE = torch.device('cpu')

print(f'Python          : {sys.version.split()[0]}')
print(f'PyTorch         : {torch.__version__}')
print(f'Platform        : {platform.processor() or platform.machine()}')
print(f'CPU count       : {os.cpu_count()}')
print(f'PyTorch threads : {torch.get_num_threads()}')
print(f'MPS available   : {torch.backends.mps.is_available()}  (not used -- see above)')
print(f'CUDA available  : {torch.cuda.is_available()}  (not used -- see above)')
print(f'Device          : {DEVICE}')
print()

if all_ok:
    print('Environment OK.')
else:
    print('Fix missing packages before continuing.')

Package availability:
  torch           2.5.1
  numpy           2.4.2
  matplotlib      3.10.8
  tqdm            4.67.3
  sympy           1.14.0

Python          : 3.11.15
PyTorch         : 2.5.1
Platform        : arm
CPU count       : 10
PyTorch threads : 4
MPS available   : True  (not used -- see above)
CUDA available  : False  (not used -- see above)
Device          : cpu

Environment OK.
